# Design & Implementation of Secure Enterprise Network Infrastructure

Welcome to the **Interactive Secure Enterprise Network Designer and Simulator (ISENDS)** project hub. 

This workbook serves as the companion guide and programmatic control panel for exploring enterprise security architectures. Along with this guide, you have access to a rich, web-based visual network topology designer and simulator served locally from this directory.

---

## 1. Core Architectural Pillars of Enterprise Network Security

When designing a secure network for an enterprise environment, we implement a **Defense-in-Depth** model where multiple layers of security controls protect intellectual property and cardholder/customer databases. Key components include:

### A. Network Segregation & VLANs (Virtual Local Area Networks)
- **VLANs** logically divide a single physical local switch into multiple isolated broadcast domains. 
- Traffic between VLANs is restricted and must be routed through a Layer 3 boundary device (like a router or firewall) to apply Access Control Lists (ACLs).

### B. The Demilitarized Zone (DMZ / Screened Subnets)
- A **DMZ** is a buffer zone containing public-facing servers (Web, DNS, VPN Gateways). 
- The DMZ prevents external clients from accessing the sensitive Internal Server/Database network directly. A compromise of the public web server does not automatically compromise internal infrastructure.

### C. Access Control Lists (ACLs) & Firewalls
- **Stateful Next-Generation Firewalls (NGFW)** inspect packets up to Layer 7.
- Access control lists define allowed sources, destinations, protocols, and ports. Rules are processed in **sequential order** (Top-to-Bottom) with an implicit **Deny All** at the end.

### D. Zero-Trust Architecture (ZTA)
- In a traditional "perimeter-based" network, anything inside the LAN is implicitly trusted. In **Zero Trust**, we verify every transaction. We use micro-segmentation so client workstations cannot communicate with high-security databases directly without active inspection, authentication, and stateful ACL checks.

### E. Packet Capture & Wireshark Protocol Analysis
- **Wireshark** is used by security analysts to intercept and decode packets traversing network segments in real time.
- **Auditing Encryption**: If databases or web services are designed insecurely, Wireshark will sniff raw plaintext payloads (such as SQL queries or credentials). Using TLS (HTTPS) or IPsec VPNs encrypts this payload, making it unreadable (scrambled cipher) to sniffers.
- **Signature Auditing**: Security teams inspect packet headers (MACs, IPs, ports, sequence numbers, flags like SYN, ACK) and hexadecimal offsets to write custom IDS/IPS filtering rules.

--- 
## 2. Launching the Interactive Web Designer & Simulator

You can run the interactive network designer directly from Python. Execute the code block below to launch the **ISENDS web application** in your browser. 

This interface will allow you to:
1. **Drag-and-drop** network nodes (Routers, NGFW Firewalls, Switch fabrics, DB Servers, Remote User VPN Gates, Clients).
2. **Connect them** to build customized trunks and links.
3. **Write policies** in the Firewall Rules list.
4. **Launch packet animations** to simulate business traffic or cyber threats (SQL Injections, Lateral Movements, Beaconing) and watch the firewall rule reactions.
5. **Audit compliance** (NIST, CIS, PCI-DSS scoreboards) and **export configurations** (Cisco IOS commands and iptables files).

In [ ]:
# Execute this cell to start the server.py locally on http://localhost:8000
import subprocess
import time
import sys
import os

print("[ISENDS] Launching Web Server (server.py) in background...")
try:
    # Locate the PEP Project directory
    pep_dir = r"C:\PEP Project"
    os.chdir(pep_dir)
    
    # Start server in the background as a subprocess
    process = subprocess.Popen([sys.executable, "server.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(2) # Give the server a moment to bind and launch browser
    print("[ISENDS] Server process started. Check your browser at: http://localhost:8000")
    print("[ISENDS] To stop the server later, interrupt/stop execution of this notebook cell.")
    
    # Monitor logs
    while process.poll() is None:
        line = process.stdout.readline().decode('utf-8').strip()
        if line:
            print(line)
        time.sleep(0.5)
except KeyboardInterrupt:
    process.terminate()
    print("\n[ISENDS] Server terminated successfully.")
except Exception as e:
    print(f"[ISENDS] Error launching server: {e}")

--- 
## 3. Programmatic Network Vulnerability Audit (Python Tool)

While the web UI provides an interactive auditor, you can also run automated python checks. The class below parses a Python dictionary representation of our topology to flag structural flaws (e.g., exposing databases directly to the WAN, or missing firewalls).

In [ ]:
class EnterpriseNetworkValidator:
    def __init__(self, nodes, edges):
        self.nodes = nodes
        self.edges = edges
        self.errors = []
        self.warnings = []
        
    def validate(self):
        self.errors = []
        self.warnings = []
        
        # Check 1: Sensitive Database Placement
        for node in self.nodes:
            if node['type'] == 'database':
                if node['zone'] == 'zone-internet':
                    self.errors.append(f"CRITICAL: Database '{node['label']}' is placed in the Internet WAN Zone!")
                elif node['zone'] == 'zone-dmz':
                    self.warnings.append(f"WARNING: Database '{node['label']}' is placed in the DMZ. Databases should reside in deep secure subnets.")
        
        # Check 2: Direct Internet-to-Internal routing (No firewall bypass)
        internet_nodes = [n['id'] for n in self.nodes if n['zone'] == 'zone-internet']
        internal_nodes = [n['id'] for n in self.nodes if n['zone'] == 'zone-internal']
        
        # Simple connectivity check to detect physical cables bypassing firewalls
        for edge in self.edges:
            src_node = next(n for n in self.nodes if n['id'] == edge['source'])
            tgt_node = next(n for n in self.nodes if n['id'] == edge['target'])
            
            if src_node['zone'] == 'zone-internet' and tgt_node['zone'] == 'zone-internal':
                if src_node['type'] != 'firewall' and tgt_node['type'] != 'firewall':
                    self.errors.append(
                        f"CRITICAL BYPASS: Link direct from WAN '{src_node['label']}' to Internal '{tgt_node['label']}' with no firewall intervening!"
                    )
                    
        # Report output
        print("=== Python Audit Scorecard ===")
        print(f"Errors found: {len(self.errors)}")
        print(f"Warnings found: {len(self.warnings)}\n")
        
        for err in self.errors:
            print(f"[✖ ERROR] {err}")
        for warn in self.warnings:
            print(f"[⚠ WARNING] {warn}")
            
        if not self.errors and not self.warnings:
            print("[✓ PASS] The network topology meets basic boundary isolation requirements!")

# Sample topology representing a compromised layout with a direct WAN database link
faulty_topology_nodes = [
    {'id': 'n1', 'label': 'WAN Router', 'type': 'internet', 'zone': 'zone-internet'},
    {'id': 'n2', 'label': 'Core Database', 'type': 'database', 'zone': 'zone-internet'},
    {'id': 'n3', 'label': 'Internal Host', 'type': 'workstation', 'zone': 'zone-internal'}
]
faulty_topology_edges = [
    {'source': 'n1', 'target': 'n2'}, # Direct WAN connection
    {'source': 'n1', 'target': 'n3'}  # Direct WAN connection to Internal Host (no firewall!)
]

auditor = EnterpriseNetworkValidator(faulty_topology_nodes, faulty_topology_edges)
auditor.validate()

--- 
## 4. Next Steps for Designing Your Infrastructure
1. **Execute the code cell** in Section 2 to start the server.
2. Open the page `http://localhost:8000` (it will open automatically in most browsers).
3. Try changing the preset to **Zero Trust DMZ Perimeter**.
4. Add a firewall rule and select **Launch Simulation**.
5. Export the Cisco configurations and copy them to analyze commands such as `ip access-group` and `permit tcp`.